<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Momentum/blob/main/The_Academic_Baseline_(12m_1m).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### The "12m-1m" Academic Baseline

Standard academic momentum (often called WML: Winners Minus Losers) was formalized by Jegadeesh and Titman in 1993.

The most critical feature of their discovery is the 1-Month Skip.

* The Trap: Stocks that spike massively in the last 20 days (e.g., due to an earnings surprise or a short squeeze) suffer from a known anomaly called short-term mean reversion. The buyers become exhausted, liquidity providers step back, and the stock often drops the following month.

* The Fix: To measure true structural momentum, we look at the 12-month return, but we deliberately cut out the most recent 1 month (21 trading days).

## Baseline Screener

In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [9]:
universe = [
    'AAPL', 'MSFT', 'AMZN', 'NVDA', 'META', 'TSLA', 'GOOGL', # Big Tech
    'JPM', 'BAC', 'GS', 'MS', 'WFC', 'C',                    # Financials
    'XOM', 'CVX', 'COP', 'EOG', 'SLB',                       # Energy
    'JNJ', 'UNH', 'LLY', 'PFE', 'ABBV',                      # Healthcare
    'PG', 'KO', 'PEP', 'WMT', 'TGT',                         # Staples/Retail
    'CAT', 'DE', 'GE', 'HON', 'MMM'                          # Industrials
]

In [10]:
# We need 252 trading days (1 year) + extra buffer for the rolling calculations
data = yf.download(universe, period="18mo")['Close']
data = data.dropna(axis=1) # Drop any stocks with missing data

[*********************100%***********************]  33 of 33 completed


In [11]:
data.head()

Ticker,AAPL,ABBV,AMZN,BAC,C,CAT,COP,CVX,DE,EOG,...,PEP,PFE,PG,SLB,TGT,TSLA,UNH,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-19,248.204178,163.049881,223.289993,41.941776,65.916023,353.507263,90.613686,132.377304,418.043854,112.317291,...,142.856903,23.251486,162.449707,35.382072,122.503807,436.170013,471.116699,66.674934,92.149117,100.419273
2024-12-20,252.874374,166.909378,224.919998,42.705582,66.657837,359.069275,90.556564,133.971664,423.785919,113.096352,...,144.101852,23.783823,161.364716,35.305382,123.489990,421.059998,481.593445,68.117859,91.004646,100.761902
2024-12-23,253.649399,169.580612,225.059998,42.434860,67.216614,358.431732,91.756119,134.084213,423.678192,113.865906,...,142.687119,24.099617,161.412750,35.717583,123.903267,430.600006,487.573273,68.311546,89.139954,101.171150
2024-12-24,256.560852,171.111115,229.050003,42.908615,68.401604,360.570160,92.451088,134.900146,424.128906,114.796974,...,144.120682,24.126684,162.209671,36.168125,124.354080,462.279999,487.342163,69.328369,91.438751,101.266335
2024-12-26,257.375610,170.350601,227.050003,43.072983,68.738800,360.128723,92.241646,135.031433,425.108765,114.397934,...,143.771744,23.964275,163.381058,36.168125,128.101639,454.130005,492.205017,69.493011,91.547279,101.351974


In [12]:
# Momentum metrics

# Constants for trading days
TRADING_DAYS_12M = 252
TRADING_DAYS_1M = 21

# Metric 1: The Naive 12-Month Return
# Formula: (Price Today / Price 252 days ago) - 1
naive_12m_return = data.pct_change(TRADING_DAYS_12M)

# Metric 2: The Academic 12m-1m Return
# Formula: (Price 21 days ago / Price 252 days ago) - 1
# We use .shift() to look backward in time
price_t_minus_1m = data.shift(TRADING_DAYS_1M)
price_t_minus_12m = data.shift(TRADING_DAYS_12M)

academic_12m_1m_return = (price_t_minus_1m / price_t_minus_12m) - 1

In [14]:
# Get the most recent valid day
latest_date = academic_12m_1m_return.dropna().index[-1]

# Extract the cross-section for that day
current_naive = naive_12m_return.loc[latest_date]
current_academic = academic_12m_1m_return.loc[latest_date]

# Combine into a DataFrame for analysis
momentum_df = pd.DataFrame({
    'Naive_12m': current_naive,
    'Academic_12m_1m': current_academic
})

# Rank the stocks based on the Academic metric (1 is the highest momentum)
momentum_df['Momentum_Rank'] = momentum_df['Academic_12m_1m'].rank(ascending=False)
momentum_df = momentum_df.sort_values('Momentum_Rank')

In [15]:
print("\n" + "="*60)
print(f" TOP 10 MOMENTUM STOCKS (As of {latest_date.date()})")
print("="*60)
# Convert to percentages for readability
display_df = momentum_df.head(10).copy()
display_df['Naive_12m'] = (display_df['Naive_12m'] * 100).round(2).astype(str) + '%'
display_df['Academic_12m_1m'] = (display_df['Academic_12m_1m'] * 100).round(2).astype(str) + '%'
print(display_df[['Academic_12m_1m', 'Naive_12m', 'Momentum_Rank']])
print("="*60)

# Check for "Momentum Traps" (Stocks where Naive is much higher than Academic)
momentum_df['Skip_Month_Return'] = (data.loc[latest_date] / data.shift(TRADING_DAYS_1M).loc[latest_date]) - 1
trap_stock = momentum_df['Skip_Month_Return'].idxmax()
trap_return = momentum_df.loc[trap_stock, 'Skip_Month_Return'] * 100

print(f"\n⚠️ The '1-Month Skip' Trap Analysis:")
print(f"The stock with the highest 1-month return was {trap_stock} (+{trap_return:.2f}% in 21 days).")
print(f"Naive momentum loves it, but academic momentum ignores this recent spike to avoid mean reversion.")


 TOP 10 MOMENTUM STOCKS (As of 2026-06-18)
       Academic_12m_1m Naive_12m  Momentum_Rank
Ticker                                         
CAT            143.08%    178.6%            1.0
GOOGL          120.82%   109.76%            2.0
SLB              60.8%    36.93%            3.0
C                58.6%    89.12%            4.0
JNJ             53.83%    53.63%            5.0
AAPL            53.42%    52.93%            6.0
NVDA             53.1%    46.39%            7.0
GS              51.01%    79.08%            8.0
MS              49.25%    75.69%            9.0
XOM             47.13%    24.74%           10.0

⚠️ The '1-Month Skip' Trap Analysis:
The stock with the highest 1-month return was GE (+25.36% in 21 days).
Naive momentum loves it, but academic momentum ignores this recent spike to avoid mean reversion.
